# Baseline Experiments — Clinical QA System
**LJMU MSc ML&AI — Amit Choudhary (1187247)**

Establishes baselines for all three experimental conditions.

In [ ]:
import sys
sys.path.insert(0, "..")
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.config import settings
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
print("Setup complete.")
print(f"  LLM: {settings.llm_model_name}")
print(f"  Embeddings: {settings.embedding_model}")

## 1. Retrieval Evaluation

In [ ]:
from src.evaluation.retrieval_eval import evaluate_retriever, compare_retrieval_strategies
print("Running retrieval evaluation...")
retrieval_metrics = evaluate_retriever(top_k_values=[1, 3, 5, 10], max_samples=100, log_to_mlflow=True)
s = retrieval_metrics.summary()
metrics_df = pd.DataFrame([
    {"Metric": "MRR",         "Value": s["mrr"]},
    {"Metric": "MAP",         "Value": s["map"]},
    {"Metric": "Recall@5",   "Value": s["recall@5"]},
    {"Metric": "Recall@10",  "Value": s["recall@10"]},
    {"Metric": "NDCG@5",     "Value": s["ndcg@5"]},
    {"Metric": "NDCG@10",    "Value": s["ndcg@10"]},
    {"Metric": "HitRate@10", "Value": s["hit_rate@10"]},
])
print(metrics_df.to_string(index=False))

In [ ]:
k_values = sorted(retrieval_metrics.recall_at_k.keys())
recalls = [retrieval_metrics.recall_at_k[k] for k in k_values]
ndcgs   = [retrieval_metrics.ndcg_at_k.get(k, 0) for k in k_values]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_values, recalls, "o-", color="steelblue", linewidth=2, markersize=8)
axes[0].set_xlabel("K"); axes[0].set_ylabel("Recall@K"); axes[0].set_title("Recall@K Curve")
axes[0].set_ylim(0, 1.0)
axes[1].plot(k_values, ndcgs, "s-", color="coral", linewidth=2, markersize=8)
axes[1].set_xlabel("K"); axes[1].set_ylabel("NDCG@K"); axes[1].set_title("NDCG@K Curve")
axes[1].set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("../docs/retrieval_curves.png", bbox_inches="tight")
plt.show()

## 2. Dense vs MMR Comparison

In [ ]:
strategy_results = compare_retrieval_strategies(max_samples=50, top_k=10)
comp = [
    {"Strategy": s.upper(), "MRR": m.mrr,
     "Recall@10": m.recall_at_k.get(10, 0),
     "NDCG@10": m.ndcg_at_k.get(10, 0)}
    for s, m in strategy_results.items()
]
print(pd.DataFrame(comp).to_string(index=False))

## 3. Prompt Template Preview

In [ ]:
from src.models.llm import CLINICAL_PROMPT_TEMPLATE
sample_ctx = "[Source: PubMed:12345]
Metformin is the first-line treatment for type 2 diabetes."
sample_q   = "What is the first-line treatment for type 2 diabetes?"
print(CLINICAL_PROMPT_TEMPLATE.format(context=sample_ctx, question=sample_q))

## 4. Load and Analyse Existing Results

In [ ]:
def load_condition(cond):
    for name in [f"e2e_{cond}_results.jsonl", f"medqa_predictions_{cond}.jsonl"]:
        p = settings.data_processed_dir / name
        if p.exists():
            return [json.loads(l) for l in open(p) if l.strip()]
    return []

conditions = ["base_llm", "rag_only", "rag_with_hallucination_filter"]
cdata = {c: load_condition(c) for c in conditions}
available = {c: d for c, d in cdata.items() if d}

if len(available) < 1:
    print("No results yet — run: python scripts/run_pipeline.py --condition all")
else:
    rows = []
    for c, d in available.items():
        rows.append({"Condition": c,
                     "Accuracy": np.mean([r.get("correct", False) for r in d]),
                     "Halluc Rate": np.mean([r.get("hallucination_rate", 0) for r in d])})
    print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
available2 = {c: d for c, d in cdata.items() if d}
if len(available2) >= 2:
    label_map = {"base_llm":"Base LLM", "rag_only":"RAG Only",
                 "rag_with_hallucination_filter":"RAG+Filter"}
    lbls   = [label_map.get(c, c) for c in available2]
    accs   = [np.mean([r.get("correct", False) for r in d]) for d in available2.values()]
    halluc = [np.mean([r.get("hallucination_rate", 0) for r in d]) for d in available2.values()]
    colors = ["#d73027", "#fee090", "#1a9850"][:len(available2)]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].bar(lbls, accs, color=colors, alpha=0.85, edgecolor="black")
    axes[0].axhline(0.75, color="red", linestyle="--", label="Target 75%")
    axes[0].set_ylabel("MedQA Accuracy"); axes[0].set_title("Accuracy by Condition")
    axes[0].set_ylim(0, 1.0); axes[0].legend()
    for i, a in enumerate(accs): axes[0].text(i, a+0.01, f"{a:.3f}", ha="center", fontweight="bold")
    axes[1].bar(lbls, halluc, color=colors, alpha=0.85, edgecolor="black")
    axes[1].set_ylabel("Hallucination Rate"); axes[1].set_title("Hallucination Rate by Condition")
    for i, h in enumerate(halluc): axes[1].text(i, h+0.005, f"{h:.3f}", ha="center", fontweight="bold")
    plt.tight_layout()
    plt.savefig("../docs/condition_comparison.png", bbox_inches="tight")
    plt.show()
else:
    print("Run pipeline first: python scripts/run_pipeline.py --condition all")